# Zemberek-NLP Turkish Dependency Parser (`ZDEP` v1) — Google Colab Trainer

This notebook trains a compact, pure-Java Turkish Universal Dependencies v2 (`UD_Turkish-BOUN`) dependency parser (`BiGRU + Deep Biaffine + Chu-Liu/Edmonds MST`) and exports a quantized `.bin.gz` model ready to load directly in `zemberek-dependency`.

**Recommended runtime:** Google Colab **T4 GPU** (free tier is sufficient).
- **Cells 1–4 (Supervised Baseline):** ~5 minutes total on T4.
- **Cells 5–7 (Optional BERTurk Teacher Distillation):** ~30–40 minutes total on T4 (`+3` to `+5` LAS boost at zero runtime cost!).

In [ ]:
#@title 1. Setup Environment (GPU check, Java 21 + Maven, Clone Repo, Python deps)
import os

!nvidia-smi

# Install JDK 21 and Maven if missing
!apt-get -qq update && apt-get -qq install -y openjdk-21-jdk maven > /dev/null
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
!java -version

# Install Python NLP deps (PyTorch is preinstalled on Colab)
!pip -q install transformers datasets

# Clone zemberek-nlp (set BRANCH to master once merged)
REPO_URL = "https://github.com/ahmetaa/zemberek-nlp.git"
BRANCH = "master"  #@param {type:"string"}
if not os.path.exists("zemberek-nlp"):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL}
else:
    !cd zemberek-nlp && git pull --ff-only

# Optional: mount Google Drive to persist checkpoints & final exported .bin.gz
MOUNT_DRIVE = False  #@param {type:"boolean"}
WORK_DIR = "/content/zdep_work"
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = "/content/drive/MyDrive/zemberek_dependency"
os.makedirs(WORK_DIR, exist_ok=True)
print("Working directory:", WORK_DIR)

In [ ]:
#@title 2. Build Zemberek & Align UD_Turkish-BOUN with Zemberek Morphology
import os

# Compile & install dependency + experiment modules locally, then build exact classpath
!cd zemberek-nlp && mvn -q -pl dependency,experiment -am install -DskipTests
!cd zemberek-nlp && mvn -q -pl experiment dependency:build-classpath -Dmdep.outputFile=/tmp/zemberek_cp.txt

with open("/tmp/zemberek_cp.txt") as f:
    dep_cp = f.read().strip()
ZEMBEREK_CP = dep_cp + ":zemberek-nlp/experiment/target/classes"
os.environ["ZEMBEREK_CP"] = ZEMBEREK_CP

# Download UD_Turkish-BOUN train / dev / test
!mkdir -p {WORK_DIR}/ud_boun {WORK_DIR}/aligned
for split in ["train", "dev", "test"]:
    url = f"https://raw.githubusercontent.com/UniversalDependencies/UD_Turkish-BOUN/master/tr_boun-ud-{split}.conllu"
    !curl -sSL {url} -o {WORK_DIR}/ud_boun/tr_boun-ud-{split}.conllu
    !java -cp "$ZEMBEREK_CP" zemberek.dependency.UdConlluAligner \
        --input {WORK_DIR}/ud_boun/tr_boun-ud-{split}.conllu \
        --output {WORK_DIR}/aligned/boun_{split}.conllu \
        --report {WORK_DIR}/aligned/audit_{split}.txt --top 10

In [ ]:
#@title 3. Train Supervised Compact ZDEP Student (~3-5 min on T4)
PRESET = "medium"  #@param ["compact", "medium", "large"]
EPOCHS = 35  #@param {type:"integer"}

!python3 zemberek-nlp/experiment/scripts/dependency/train_student.py \
    --train {WORK_DIR}/aligned/boun_train.conllu \
    --dev   {WORK_DIR}/aligned/boun_dev.conllu \
    --test  {WORK_DIR}/aligned/boun_test.conllu \
    --preset {PRESET} \
    --epochs {EPOCHS} \
    --out-dir {WORK_DIR}/student_supervised_{PRESET}

In [ ]:
#@title 4. Verify Exact PyTorch <-> Pure-Java Parity & Benchmark Speed in Java
STUDENT_DIR = f"{WORK_DIR}/student_supervised_{PRESET}"

print("--- FP32 Parity Check (should be ~100.0% match vs PyTorch) ---")
!java -cp "$ZEMBEREK_CP" zemberek.dependency.EvaluateDependencyModel \
    --model {STUDENT_DIR}/dep_boun_{PRESET}_fp32.bin.gz \
    --test  {WORK_DIR}/aligned/boun_test.conllu \
    --expected {STUDENT_DIR}/pytorch_test_preds.conllu \
    --passes 3

print("\n--- INT8 Production Model Evaluation + Live Sample Parse ---")
!java -cp "$ZEMBEREK_CP" zemberek.dependency.EvaluateDependencyModel \
    --model {STUDENT_DIR}/dep_boun_{PRESET}_int8.bin.gz \
    --test  {WORK_DIR}/aligned/boun_test.conllu \
    --expected {STUDENT_DIR}/pytorch_test_preds.conllu \
    --sentence "Prof. Dr. Fethi Bey saat 14.30'da Türkiye'ye döndü." \
    --passes 3

---
## Optional Phase 4: BERTurk Teacher–Student Distillation (+3 to +5 LAS boost!)

Run Cells 5–7 to fine-tune `dbmdz/bert-base-turkish-cased` on `UD_Turkish-BOUN`, label ~60k–100k clean Turkish Wikipedia & UD sentences, and distill them into the compact pure-Java `ZDEP` student.

In [ ]:
#@title 5. Train BERTurk Deep Biaffine Teacher (~12 min on T4)
TEACHER_BACKBONE = "dbmdz/bert-base-turkish-cased"  #@param ["dbmdz/bert-base-turkish-cased", "dbmdz/convbert-base-turkish-cased"]

!python3 zemberek-nlp/experiment/scripts/dependency/train_teacher_distill.py train \
    --train {WORK_DIR}/aligned/boun_train.conllu \
    --dev   {WORK_DIR}/aligned/boun_dev.conllu \
    --test  {WORK_DIR}/aligned/boun_test.conllu \
    --backbone {TEACHER_BACKBONE} \
    --epochs 15 \
    --batch-size 24 \
    --out {WORK_DIR}/teacher_berturk.pt

In [ ]:
#@title 6. Collect Unlabeled Turkish Sentences, Project via Zemberek & Label with Teacher
MAX_UNLABELED = 60000  #@param {type:"integer"}
MIN_CONF = 0.88  #@param {type:"number"}

from datasets import load_dataset

# Stream clean Turkish Wikipedia paragraphs (no giant disk download)
ds = load_dataset("wikimedia/wikipedia", "20231101.tr", split="train", streaming=True)
raw_txt = f"{WORK_DIR}/unlabeled_tr.txt"
written = 0
with open(raw_txt, "w", encoding="utf-8") as f:
    for row in ds:
        for para in row["text"].split("\n"):
            p = para.strip()
            if 40 <= len(p) <= 600 and not p.startswith("=="):
                f.write(p + "\n")
                written += 1
                if written >= MAX_UNLABELED:
                    break
        if written >= MAX_UNLABELED:
            break
print(f"Collected {written} Turkish paragraphs -> {raw_txt}")

# Project paragraphs to UD v2 CoNLL-U via Zemberek (with sentence segmentation)
!java -cp "$ZEMBEREK_CP" zemberek.dependency.UdConlluAligner \
    --project-text {raw_txt} \
    --segment \
    --max-sentences {MAX_UNLABELED} \
    --output {WORK_DIR}/unlabeled_projected.conllu

# Label with BERTurk teacher & filter by mean arc confidence
!python3 zemberek-nlp/experiment/scripts/dependency/train_teacher_distill.py label \
    --checkpoint {WORK_DIR}/teacher_berturk.pt \
    --input  {WORK_DIR}/unlabeled_projected.conllu \
    --output {WORK_DIR}/silver_distill.conllu \
    --min-conf {MIN_CONF}

In [ ]:
#@title 7. Train Distilled ZDEP Student & Verify Final Pure-Java INT8 Model
DISTILL_DIR = f"{WORK_DIR}/student_distilled_{PRESET}"

!python3 zemberek-nlp/experiment/scripts/dependency/train_student.py \
    --train  {WORK_DIR}/aligned/boun_train.conllu \
    --dev    {WORK_DIR}/aligned/boun_dev.conllu \
    --test   {WORK_DIR}/aligned/boun_test.conllu \
    --silver {WORK_DIR}/silver_distill.conllu \
    --silver-weight 0.6 \
    --gold-repeat 3 \
    --preset {PRESET} \
    --epochs 25 \
    --out-dir {DISTILL_DIR}

!java -cp "$ZEMBEREK_CP" zemberek.dependency.EvaluateDependencyModel \
    --model {DISTILL_DIR}/dep_boun_{PRESET}_int8.bin.gz \
    --test  {WORK_DIR}/aligned/boun_test.conllu \
    --expected {DISTILL_DIR}/pytorch_test_preds.conllu \
    --sentence "Prof. Dr. Fethi Bey saat 14.30'da Türkiye'ye döndü." \
    --passes 3

# Download final INT8 model directly to your browser
try:
    from google.colab import files
    files.download(f"{DISTILL_DIR}/dep_boun_{PRESET}_int8.bin.gz")
except ImportError:
    print("Exported model ready at:", f"{DISTILL_DIR}/dep_boun_{PRESET}_int8.bin.gz")